## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:梁臣京


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys
from typing import List

# 全局变量
ans = []
o = []
posi = []
a = 0
b = 0
l = 0
n = 0

def refresh():
    """更新位置映射"""
    global posi, o, n
    for i in range(n):
        posi[o[i]] = i

def doMagic():
    """执行交换魔法"""
    global ans, o, a, b, n
    ans.append(0)
    for i in range(n):
        if o[i] == a:
            o[i] = b
        elif o[i] == b:
            o[i] = a
    refresh()

def doAdd(x):
    """执行加法魔法"""
    global ans, o, n
    if x == 0:
        return
    ans.append(x)
    for i in range(n):
        o[i] = (o[i] + x) % n
    refresh()

def doXor(x):
    """执行异或魔法"""
    global ans, o
    if x == 0:
        return
    ans.append(-x)
    for i in range(n):
        o[i] = o[i] ^ x
    refresh()

def calc(a_val, b_val):
    """计算pa和pb"""
    global l, n
    delta = (b_val - a_val + n - l + n) % n
    pa = pb = 0
    
    stp = n // 2
    while stp >= 2 * l:
        if delta >= stp:
            delta -= stp
            pb += stp // 2
        else:
            pa += stp // 2
        stp //= 2
    
    pa += n // 2
    pa += (a_val & (l - 1))
    pb += (a_val & (l - 1))
    
    return pa, pb

def doSwap(c, d):
    """执行交换操作"""
    global a, b, l, n
    
    if (c // l) % 2 == (d // l) % 2:
        if (c // l) % 2 == 0:
            p = (c & (l - 1)) + l
        else:
            p = (c & (l - 1))
        doSwap(c, p)
        doSwap(d, p)
        doSwap(c, p)
    else:
        pa, pb = calc(a, b)
        pc, pd = calc(c, d)
        
        doAdd((pc - c + n) % n)
        doXor(pc ^ pa)
        doAdd((a - pa + n) % n)
        doMagic()
        doAdd((pa - a + n) % n)
        doXor(pc ^ pa)
        doAdd((c - pc + n) % n)

class Perm:
    """排列处理类"""
    def __init__(self):
        self.a = []
        self.n = 0
        self.vec = []
    
    def __lt__(self, other):
        """比较运算符"""
        for i in range(self.n):
            if self.a[i] != other.a[i]:
                return self.a[i] < other.a[i]
        return False
    
    def print_perm(self):
        """打印排列"""
        print(' '.join(str(x) for x in self.a))
    
    def inv(self):
        """求逆排列"""
        res = Perm()
        res.a = [0] * self.n
        for i in range(self.n):
            res.a[self.a[i]] = i
        res.n = self.n
        return res
    
    def sorted(self):
        """检查是否已排序"""
        for i in range(self.n - 1):
            if self.a[i] > self.a[i + 1]:
                return False
        return True
    
    def calc(self):
        """递归计算"""
        f = [False] * 100005
        for i in range(self.n):
            f[i] = True
        
        for i in range(self.n):
            if not f[i]:
                return False
        
        if self.n == 1:
            return True
        
        b = Perm()
        c = Perm()
        
        for i in range(self.n // 2):
            b.a.append(self.a[i * 2] // 2)
            c.a.append(self.a[i * 2 + 1] // 2)
        
        b.n = self.n // 2
        c.n = self.n // 2
        
        if not b.calc() or not c.calc():
            return False
        
        if self.a[0] % 2:
            self.vec.append(self.n == 2 and 1 or -1)
        
        tb = 0
        for i in b.vec:
            if i > 0:
                self.vec.append(-1)
                self.vec.append(1)
            else:
                self.vec.append(i * 2)
                tb ^= -i * 2
        
        if tb:
            self.vec.append(-tb)
        
        tc = 0
        for i in c.vec:
            if i > 0:
                self.vec.append(1)
                self.vec.append(-1)
            else:
                self.vec.append(i * 2)
                tc ^= -i * 2
        
        if (tc & (self.n // 2)) != (tb & (self.n // 2)):
            for i in range(self.n // 4):
                self.vec.append(-1)
                self.vec.append(1)
        
        if tb >= (self.n // 2):
            tb -= self.n // 2
        if tc >= (self.n // 2):
            tc -= self.n // 2
        
        if tb != tc:
            return False
        
        tmp = []
        for i in self.vec:
            if not tmp:
                tmp.append(i)
            else:
                if i < 0 and tmp[-1] < 0:
                    tmp[-1] = -((-tmp[-1]) ^ (-i))
                    if tmp[-1] == 0:
                        tmp.pop()
                else:
                    tmp.append(i)
        
        self.vec = tmp
        return True

def main():
    """主函数"""
    global ans, o, posi, a, b, l, n
    
    # 读取输入
    data = sys.stdin.read().strip().split()
    if not data:
        return
    
    it = iter(data)
    n = int(next(it))
    a = int(next(it))
    b = int(next(it))
    
    # 初始化数组
    o = [int(next(it)) for _ in range(n)]
    posi = [0] * n
    
    refresh()
    
    # 计算l
    l = (a - b + n) % n
    l = l & -l
    if l == 0:
        l = n
    
    # 预处理
    if l > 1:
        perm_a = Perm()
        perm_a.n = l
        for i in range(n):
            perm_a.a.append(o[i] & (l - 1))
        
        if not perm_a.calc():
            print("-1")
            return
        
        for val in perm_a.vec:
            if val > 0:
                doAdd(val)
            else:
                doXor(-val)
    
    # 处理每个剩余类
    for i in range(l):
        vec = []
        for j in range(i, n, l):
            vec.append(o[j])
        
        c = 0
        flag = 1
        vec_sorted = sorted(vec)
        
        for j in range(i, n, l):
            if vec_sorted[c] != j:
                flag = 0
                break
            c += 1
        
        if not flag:
            print("-1")
            return
        
        for j in range(i, n, l):
            if o[j] != j:
                doSwap(j, o[j])
    
    # 验证结果
    for i in range(n):
        assert o[i] == i, f"Position {i} has value {o[i]}, expected {i}"
    
    # 输出答案
    print(len(ans))
    for val in ans:
        if val == 0:
            print("0")
        elif val < 0:
            print(f"1 {-val}")
        else:
            print(f"2 {val}")

if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys
import heapq

def find_solution(city_count, road_length, max_energy, initial_supply):
    fuel_stations = {}
    for location, cost in stations_list:
        if location in fuel_stations:
            fuel_stations[location] = min(fuel_stations[location], cost)
        else:
            fuel_stations[location] = cost
    
    priority_queue = [(-max_energy, -initial_supply, 0)]
    visited_states = set()
    
    while priority_queue:
        neg_current_energy, neg_current_coins, current_position = heapq.heappop(priority_queue)
        current_energy, current_coins = -neg_current_energy, -neg_current_coins
        
        if current_position + current_energy >= road_length:
            return "Yes"
        
        for next_pos in range(current_position + 1, min(current_position + current_energy + 1, road_length)):
            if next_pos in fuel_stations and current_coins >= fuel_stations[next_pos]:
                updated_coins = current_coins - fuel_stations[next_pos]
                
                if (next_pos, updated_coins) not in visited_states:
                    visited_states.add((next_pos, updated_coins))
                    heapq.heappush(priority_queue, (-max_energy, -updated_coins, next_pos))
    return "No"

input_data = sys.stdin.read().strip().split("\n")
index = 0
results = []

while index < len(input_data):
    city_count, road_length, max_energy, initial_supply = map(int, input_data[index].split())
    index += 1
    stations_list = []
    
    for _ in range(city_count):
        location, cost = map(int, input_data[index].split())
        stations_list.append((location, cost))
        index += 1
    
    results.append(find_solution(city_count, road_length, max_energy, initial_supply))

print("\n".join(results))

## C 最长回文

In [ ]:
#include<iostream>
#include<string.h>
#include<algorithm>
#include<math.h>
#include<vector>
#include<map>
#include<unordered_map>
#include<queue>
#include<set> 
#include<stack>
#include<bitset>
#define endl '\n'
using namespace std;
typedef long long ll;
typedef pair<int ,int > PII;
typedef unsigned long long ull;
const int N=2e5+5;
int n;
int len_pal_a[N],len_pal_b[N];
char str_a[N],str_b[N];
string input_a,input_b;

void find_palindromes(string original_str, char *processed_str, int *pal_lengths){
    int index = 0;
    processed_str[index++]='(';
    processed_str[index++]='#';
    for(int i=0;i<original_str.size();i++){
        processed_str[index++]=original_str[i];
        processed_str[index++]='#';
    }
    processed_str[index++]='!';
    int max_right=0,mid_point;
    for(int i=1;i<index;i++){
        if(max_right>i) pal_lengths[i]=min(pal_lengths[2*mid_point-i],max_right-i);
        else pal_lengths[i]=1;
        while(processed_str[i+pal_lengths[i]]==processed_str[i-pal_lengths[i]])pal_lengths[i]++;
        if(i+pal_lengths[i]>max_right){
            max_right=i+pal_lengths[i];
            mid_point=i;
        }
    }
    
}

int main(){
	ios::sync_with_stdio(false);cin.tie(0);cout.tie(0);
	cin>>n;
    cin>>input_a>>input_b;
    find_palindromes(input_a,str_a,len_pal_a);
    find_palindromes(input_b,str_b,len_pal_b);
    int result=1;
    for(int i=2;i<=n*2+1;i++){
        int temp_len=max(len_pal_a[i],len_pal_b[i-2]);
        while(str_a[i-temp_len]==str_b[i+temp_len-2])temp_len++;
        result=max(result,temp_len);
    }
    cout<<result-1;
}

## D 优惠券

In [ ]:
import sys
import bisect

maxn = int(5e5) + 9

def solve():
    global m, a, vis, la, t, s
    flag = True
    ans = -1
    vis = [0] * maxn
    la = [0] * maxn
    s = []
    
    for i in range(1, m + 1):
        data = sys.stdin.readline().split()
        if not data:
            continue
        t = data[0]
        
        if t == "I" or t == "O":
            if len(data) < 2:
                continue
            a = int(data[1])
            if not flag:
                continue
                
            if t == "I":
                vis[a] += 1
            else:
                vis[a] -= 1
                
            if vis[a] < 0 or vis[a] > 1:
                pos = bisect.bisect_left(s, la[a])
                if pos == len(s):
                    ans = i
                    flag = False
                else:
                    s.pop(pos)
                    vis[a] = max(0, min(vis[a], 1))
            la[a] = i
        else:
            bisect.insort(s, i)
    
    print(ans)

def main():
    global m
    for line in sys.stdin:
        if line.strip():
            m = int(line.strip())
            solve()

if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
def find_parent(parent, x):
    if parent[x] != x:
        parent[x] = find_parent(parent, parent[x])
    return parent[x]

def union(parent, rank, x, y):
    px, py = find_parent(parent, x), find_parent(parent, y)
    if px == py:
        return
    if rank[px] < rank[py]:
        px, py = py, px
    parent[py] = px
    if rank[px] == rank[py]:
        rank[px] += 1

n = int(input())
points = []
for i in range(n):
    x, y = map(int, input().split())
    points.append((x, y))

# 初始化并查集
parent = list(range(n))
rank = [0] * n

# 将同一行或同一列的点合并
for i in range(n):
    for j in range(i + 1, n):
        # 如果在同一行或同一列
        if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
            union(parent, rank, i, j)

# 统计连通分量数量
components = set()
for i in range(n):
    components.add(find_parent(parent, i))

# 需要添加的点数 = 连通分量数量 - 1
print(len(components) - 1)

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct SegInfo {
    string raw;       
    int len;          

    vector<pair<int, int>> parts;
};

vector<SegInfo> segs;  
int prefixLen, suffixLen;


void preprocess(const string& p) {
    
    vector<string> rawSegs;
    string cur;
    for (char c : p) {
        if (c == '*') {
            rawSegs.push_back(cur);
            cur.clear();
        } else {
            cur += c;
        }
    }
    rawSegs.push_back(cur);

    prefixLen = rawSegs[0].size();
    suffixLen = rawSegs.back().size();

    for (const string& s : rawSegs) {
        SegInfo info;
        info.raw = s;
        info.len = s.size();

        
        vector<pair<int, int>> parts;
        int i = 0;
        while (i < (int)s.size()) {
            if (s[i] == '?') { i++; continue; }
            int j = i;
            while (j < (int)s.size() && s[j] != '?') j++;
            parts.push_back({i, j - i});
            i = j;
        }
        
        sort(parts.begin(), parts.end(),
             [](const pair<int, int>& a, const pair<int, int>& b) {
                 return a.second > b.second;
             });
        info.parts = parts;
        segs.push_back(info);
    }
}


bool segVerify(int segIdx, const string& s, int pos) {
    const SegInfo& info = segs[segIdx];
    if (pos + info.len > (int)s.size()) return false;
    
   
    for (auto& p : info.parts) {
        int off = p.first, ln = p.second;
        if (s.compare(pos + off, ln, info.raw, off, ln) != 0) 
            return false;
    }
    

    return true;
}


int findSeg(int segIdx, const string& s, int start, int maxEnd) {
    const SegInfo& info = segs[segIdx];
    int m = info.len;
    if (m == 0) return start;  
    if (start + m > maxEnd) return -1;

    
    if (info.parts.empty())
        return start + m <= maxEnd ? start + m : -1;

    int limit = maxEnd - m;

    
    int anchorOff = info.parts[0].first;
    int anchorLen = info.parts[0].second;
    string anchor = info.raw.substr(anchorOff, anchorLen);

   
    bool hasSec = (info.parts.size() >= 2);
    int secOff = hasSec ? info.parts[1].first : 0;
    int secLen = hasSec ? info.parts[1].second : 0;

    size_t searchFrom = start + anchorOff;  
    while (true) {
        if ((int)searchFrom > limit + anchorOff) break;  
        size_t found = s.find(anchor, searchFrom);
        if (found == string::npos || (int)found > limit + anchorOff) break;

        int candidate = (int)found - anchorOff;
        
        if (candidate < start || candidate > limit) {
            searchFrom = found + 1;
            continue;
        }

       
        if (hasSec) {
            
            if (candidate + secOff + secLen > (int)s.size() ||
                s.compare(candidate + secOff, secLen, info.raw, secOff, secLen) != 0) {
                searchFrom = found + 1;
                continue;
            }
        }

        
        if (segVerify(segIdx, s, candidate)) {
            return candidate + m;
        }

        searchFrom = found + 1;
    }

    return -1;
}

bool match(const string& filename) {
    int n = filename.size();

    
    if (segs.size() == 1) {
        return segs[0].len == n && segVerify(0, filename, 0);
    }

    
    if (prefixLen > 0) {
        if (n < prefixLen || !segVerify(0, filename, 0))
            return false;
    }

    
    int lastIdx = (int)segs.size() - 1;
    if (suffixLen > 0) {
        if (n < suffixLen || !segVerify(lastIdx, filename, n - suffixLen))
            return false;
    }

    int pos = prefixLen;
    int suffixStart = n - suffixLen;

    
    for (int i = 1; i < lastIdx; i++) {
        if (segs[i].len == 0) continue; 
        int end = findSeg(i, filename, pos, suffixStart);
        if (end == -1) return false;
        pos = end;
    }

    
    return pos <= suffixStart;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;
    preprocess(pattern);

    int n;
    cin >> n;
    while (n--) {
        string filename;
        cin >> filename;
        cout << (match(filename) ? "YES" : "NO") << '\n';
    }

    return 0;
}

## G 汉诺塔

In [ ]:
import sys

sys.setrecursionlimit(2000)

def solve():
    line1 = sys.stdin.readline()
    if not line1: return
    n = int(line1.strip())
    priorities = sys.stdin.readline().strip().split()

    priority_map = {}
    for i, op in enumerate(priorities):
        priority_map[op] = i

    memo = {}

    def get_steps(count, src, dst):
        if count == 0: return 0
        key = (count, src, dst)
        if key in memo: return memo[key]

        aux = 3 - src - dst
        
        ops = [(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]
        valid_moves = []
        for s, d in ops:
            op_str = chr(ord('A') + s) + chr(ord('A') + d)
            if op_str in priority_map:
                valid_moves.append((priority_map[op_str], s, d))
        
        valid_moves.sort()

        target_for_big = -1
        for _, s, d in valid_moves:
            if s == src:
                target_for_big = d
                break

        if target_for_big == dst:
            res = get_steps(count - 1, src, aux) + 1 + get_steps(count - 1, aux, dst)
        else:
            res = get_steps(count - 1, src, dst) + 1 + get_steps(count - 1, dst, aux)
            
        memo[key] = res
        return res

    valid_all = []
    for s, d in [(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]:
        op_str = chr(ord('A') + s) + chr(ord('A') + d)
        if op_str in priority_map:
            valid_all.append((priority_map[op_str], s, d))
    valid_all.sort()

    final_dst = -1
    for _, s, d in valid_all:
        if s == 0:
            final_dst = d
            break

    print(get_steps(n, 0, final_dst))

solve()

## H 马步距离

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    x1, y1, x2, y2 = map(int, input_data)

    dx = abs(x1 - x2)
    dy = abs(y1 - y2)

    if dx < dy:
        dx, dy = dy, dx

    if dx > 2 * dy:
        res = (dx + 1) // 2
    else:
        res = (dx + dy) // 3

    if (dx + dy) % 3 != 0:
        res += 1

    if dx == 1 and dy == 0:
        res = 3
    elif dx == 2 and dy == 2:
        res = 4

    print(res)

if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        n = len(heights)
        if n == 0:
            return 0
        
        stack = []  # 单调递增栈，存索引
        max_area = 0
        
        # 遍历 i 从 0 到 n（n 是虚拟位置，对应高度 0）
        for i in range(n + 1):
            # 当前高度：i == n 时为 0，否则为 heights[i]
            curr_h = 0 if i == n else heights[i]
            
            # 弹出所有比当前高度高的柱子
            while stack and curr_h < heights[stack[-1]]:
                # 弹出栈顶
                top_idx = stack.pop()
                h = heights[top_idx]
                
                # 左边界：弹出后栈顶是左边第一个比 h 小的索引；若栈空，则左边界为 -1
                left_bound = stack[-1] if stack else -1
                
                # 宽度 = 右边界(i) - 左边界 - 1
                width = i - left_bound - 1
                
                area = h * width
                if area > max_area:
                    max_area = area
            
            stack.append(i)
        
        return max_area

## J 消防局的设立

In [ ]:
import sys
from collections import deque

def main():
    data = sys.stdin.read().strip().split()
    if not data:
        print(0)
        return
    n = int(data[0])
    graph = [[] for _ in range(n+1)]
    
    # 读入 n-1 条边：第 i 行（i从1到n-1）表示边 (i+1, p)
    for i in range(1, n):
        p = int(data[i])
        u, v = i+1, p
        graph[u].append(v)
        graph[v].append(u)
    
    # BFS 求深度和父节点（根为1）
    depth = [-1] * (n+1)
    parent = [0] * (n+1)
    depth[1] = 0
    q = deque([1])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if depth[v] == -1:
                depth[v] = depth[u] + 1
                parent[v] = u
                q.append(v)
    
    # 按深度降序
    nodes = sorted(range(1, n+1), key=lambda x: depth[x], reverse=True)
    covered = [False] * (n+1)
    ans = 0

    # 从某点出发，标记距离<=2的所有点
    def mark_cover(center):
        nonlocal covered
        q = deque()
        q.append((center, 0))
        vis = [False] * (n+1)
        vis[center] = True
        while q:
            u, d = q.popleft()
            if d > 2:
                continue
            covered[u] = True
            for v in graph[u]:
                if not vis[v]:
                    vis[v] = True
                    q.append((v, d+1))

    for u in nodes:
        if not covered[u]:
            # 向上跳2步
            x = u
            for _ in range(2):
                if parent[x] != 0:
                    x = parent[x]
                else:
                    break
            ans += 1
            mark_cover(x)
    
    print(ans)

if __name__ == "__main__":
    main()